In [1]:
from IPython.display import Image

- 上下文窗口 (Context Window)：这是LLM模型本身能够一次性处理信息的“工作记忆”。它的大小是固定的，所有实时的推理和生成都必须在这个有限的空间内完成。这是模型架构的根本限制。
- 外部记忆库 (External Memory Bank)：Memory-R1为模型配备了一个独立的外部数据库。这个记忆库可以存储海量的、跨越长时间对话的信息，其容量远超上下文窗口的限制。

In [4]:
Image(url='https://blog.langchain.com/content/images/size/w1600/2025/07/image-6.png', width=500)

### Generative Agents

In [3]:
Image(url='./figs/generative_agent_arch.png', width=500)

- longer-term plans and create higher-level reflections
    - from `raw observational memory` to  reflections (必要性)
        - 仅有观察记忆的智能体难以进行深层推理。 论文中举了一个例子：当被问及最想和谁待在一起时，一个没有反思能力的智能体Klaus仅凭互动频率选择了邻居Wolfgang，尽管他们的交流很肤浅。 而通过反思，Klaus能够意识到自己对研究充满热情，并发现Maria也在从事研究，从而认识到他们有共同兴趣，最终选择了Maria。
        - We introduce a second type of memory, which we call a reflection. (as a type of memory)
            - 触发时机：智能体感知到的最新事件的重要性分数累计超过一定阈值（论文中设为150）时被触发，大约每天发生两到三次。
        - 如何生成问题
            - 触发后，系统会提取最近的100条记忆，并用大型语言模型提问：“Given only the information above, what are 3 most salient high-level questions we can answer about the subjects in the statements?” (仅根据以上信息，关于陈述中的主题，我们能回答哪3个最突出的高层次问题？) 这会生成一些候选问题
                - What topic is Klaus Mueller passionate about?
                -  What is the relationship between Klaus Mueller and Maria Lopez?
        -  收集证据与综合: 系统将每个生成的问题作为查询，去检索相关的记忆（包括过去的观察和其他反思）。 然后，将这些检索到的记忆作为证据，再次提示语言模型来提炼出高级见解，并要求其引用证据来源。
            -  Statements about Klaus Mueller
                -  Klaus Mueller is writing a research paper
                -  Klaus Mueller enjoys reading a book on gentrification [...]
            -  What 5 high-level insights can you infer from the above statements? (example format: insight (because of 1, 5, 3))
                -  `Klaus Mueller is dedicated to his research on gentrification (because of 1, 2, 8, 15)`

### Memory Management

- store, update, retrieve
    - structured memory operations, including adding, updating, deleting, or taking no operation on memory entries;
        - Add, Update, Delete, Noop
- Memory-R1
    - `“安德鲁领养了一只叫巴迪的狗”和后来的“我又领养了一只叫斯考特的狗”整合成一条记忆：“安德鲁有两只狗，分别叫巴迪和斯考特”。`